# Grape Disease Classification

In [ ]:
import pathlib
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt
import numpy as np

BATCH_SIZE = 32
IMAGE_SIZE = 256
CHANNELS   = 3
EPOCHS     = 50
# Find the Crop-Care-AI project root regardless of where the kernel's CWD is.
# The project root always contains both "training/" and "api/" subdirectories.
_cwd = pathlib.Path().resolve()
_project_root = None
for _p in [_cwd] + list(_cwd.parents):
    if (_p / "training").is_dir() and (_p / "api").is_dir():
        _project_root = _p
        break
if _project_root is None:
    raise RuntimeError(f"Could not locate project root from CWD: {_cwd}")

DATA_DIR = str(_project_root.parent / "Data" / "Grape")
print(f"Project root : {_project_root}")
print(f"DATA_DIR     : {DATA_DIR}")

In [ ]:
# Load pre-split datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR + "/Train", seed=123, shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR + "/Val", seed=123, shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE)
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR + "/Test", seed=123, shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
print("Classes:", class_names)
print(f"Train batches: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}")

In [ ]:
# Visualize sample images
plt.figure(figsize=(12, 12))
for image_batch, labels_batch in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(image_batch[i].numpy().astype("uint8"))
        plt.title(class_names[labels_batch[i]])
        plt.axis("off")

In [ ]:
# Preprocess: cache, shuffle, prefetch
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds  = test_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

# Resizing, normalization and augmentation layers
resize_and_rescale = tf.keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Rescaling(1./255),
])
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
])
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
).prefetch(buffer_size=tf.data.AUTOTUNE)

# Build model
n_classes    = len(class_names)
input_shape  = (BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)

model = tf.keras.Sequential([
    resize_and_rescale,
    layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes, activation='softmax'),
])
model.build(input_shape=input_shape)
model.summary()

# Compile and train
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])
history = model.fit(train_ds, batch_size=BATCH_SIZE, validation_data=val_ds,
                    verbose=1, epochs=EPOCHS)

# Evaluate
scores = model.evaluate(test_ds)
print(f"Test Loss: {scores[0]:.4f}  |  Test Accuracy: {scores[1]*100:.2f}%")

# Plot accuracy and loss
hist = dict(history.history)
acc, val_acc = hist['accuracy'], hist['val_accuracy']
loss, val_loss = hist['loss'], hist['val_loss']
epochs_range = range(len(acc))
plt.figure(figsize=(8, 8))
plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Train Accuracy')
plt.plot(epochs_range, val_acc, label='Val Accuracy')
plt.legend(); plt.title('Accuracy')
plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.legend(); plt.title('Loss')
plt.show()

# Predict on sample test images
def predict(model, img):
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    predictions = model.predict(img_array)
    return class_names[np.argmax(predictions[0])], round(100 * np.max(predictions[0]), 2)

plt.figure(figsize=(15, 15))
for images, labels in test_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        pred_class, confidence = predict(model, images[i].numpy())
        plt.title(f"Actual: {class_names[labels[i]]}\nPred: {pred_class}\nConf: {confidence}%")
        plt.axis("off")

# Save
import os
os.makedirs("../saved_models", exist_ok=True)
model_version = max([int(i) for i in os.listdir("../saved_models") + [0]]) + 1
model.export(f"../saved_models/{model_version}")
model.save("../grape.keras")
print(f"Saved: ../saved_models/{model_version}  and  ../grape.keras")